# ECT: Interactive Rotation Curves

**Paper Fig. 1, §6** | Valeriy Blagovidov, *Euclidean Condensate Theory* (2026)

**φ-first IR approximation** — the effective gravitational coupling follows from the condensate
φ-field profile. In the IR limit (flat condensate, large *r*) this reduces to:

$$G_{\rm eff}(r)\approx G_N\sqrt{1+(r/r_0)^2}$$

where $r_0$ is the condensate scale length (one free parameter per galaxy).

> **Note:** This is the IR approximation of the full φ-first closure. The complete
> φ-branch fit (all 175 SPARC galaxies) is in `ect_sparc_fit_phi_branch.py`.
> Results are in `ect_sparc_results_v3.csv`.

**ECT prediction:** $r_0 \propto M_\star^{1/3}$ (zero-parameter exponent, from dimensional
analysis of the condensate action — **Level A**).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
%matplotlib inline

G = 4.302e-6  # (km/s)^2 kpc/Msun

# Representative SPARC galaxies (schematic from paper fits)
# Full fits: ect_sparc_results_v3.csv, ect_sparc_fit_phi_branch.py
galaxies = {
    'NGC 3198  (r0_best=6.7 kpc)': {
        'r': np.array([1,2,3,4,5,6,7,8,9,10,12,14,16,18,20,22,25,28,30]),
        'v_obs': np.array([100,120,132,138,142,145,147,148,149,149,150,150,150,149,149,148,148,148,148]),
        'v_err': np.full(19, 4.0), 'r0_best': 6.7, 'chi2': 1.38, 'logM': 10.54},
    'NGC 2403  (r0_best=2.3 kpc)': {
        'r': np.array([0.5,1,1.5,2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
        'v_obs': np.array([60,80,95,105,117,122,125,127,129,130,131,132,133,133,133,132,131]),
        'v_err': np.full(17, 4.0), 'r0_best': 2.3, 'chi2': 1.40, 'logM': 9.65},
    'DDO 154   (r0_best=0.10 kpc)': {
        'r': np.array([0.1,0.3,0.6,0.9,1.2,1.6,2.0,2.5,3.0,3.5,4.0]),
        'v_obs': np.array([12,18,23,27,30,33,36,38,40,41,42]),
        'v_err': np.full(11, 3.0), 'r0_best': 0.10, 'chi2': 3.55, 'logM': 7.47},
}

def plot_galaxy(galaxy_name, r0_kpc):
    gal = galaxies[galaxy_name]
    r, v_obs, v_err = gal['r'], gal['v_obs'], gal['v_err']
    r0_ref = gal['r0_best']
    # Reconstruct approximate Newtonian curve from best-fit ECT
    v_newton = v_obs / (1 + (r/r0_ref)**2)**0.25
    v_ect = v_newton * (1 + (r/r0_kpc)**2)**0.25
    chi2 = np.mean(((v_obs - v_ect)/v_err)**2)

    r_fine = np.linspace(0.01, r.max()*1.1, 300)
    vn_fine = np.interp(r_fine, r, v_newton)
    vect_fine = vn_fine * (1 + (r_fine/r0_kpc)**2)**0.25

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.errorbar(r, v_obs, yerr=v_err, fmt='ko', ms=5, capsize=3, label='SPARC data')
    ax1.plot(r_fine, vn_fine, 'b--', lw=1.5, label='Newtonian (baryons only)')
    ax1.plot(r_fine, vect_fine, 'g-', lw=2.5,
             label=f'ECT IR approx  r₀={r0_kpc:.2f} kpc')
    ax1.axvline(r0_kpc, color='g', ls=':', alpha=0.5)
    ax1.set_xlabel('r [kpc]'); ax1.set_ylabel('v [km/s]')
    ax1.set_title(f'{galaxy_name}  |  χ²/N={chi2:.2f}  (paper best: {gal["chi2"]})')
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    Geff = np.sqrt(1 + (r_fine/r0_kpc)**2)
    ax2.plot(r_fine, Geff, 'g-', lw=2)
    ax2.axvline(r0_kpc, color='g', ls=':', alpha=0.5, label=f'r₀={r0_kpc:.2f} kpc')
    ax2.axhline(np.sqrt(2), color='gray', ls='--', alpha=0.5,
                label=r'$\sqrt{2}$ at r=r₀')
    ax2.set_xlabel('r [kpc]')
    ax2.set_ylabel(r'$G_{\rm eff}(r)/G_N$  [IR approximation]')
    ax2.set_title('Effective gravitational coupling')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f'Paper best r₀: {r0_ref} kpc | Current: {r0_kpc:.2f} kpc | χ²/N: {chi2:.2f}')
    print(f'log M*/M⊙ = {gal["logM"]}  |  ECT: r0 ∝ M*^(1/3)  [Level A]')

interact(plot_galaxy,
         galaxy_name=Dropdown(options=list(galaxies.keys()), description='Galaxy:'),
         r0_kpc=FloatSlider(value=6.7, min=0.05, max=25.0, step=0.05,
                             description='r₀ [kpc]:', continuous_update=False,
                             style={'description_width': 'initial'}));

## r₀ vs M★ scaling — ECT Level A prediction

The condensate action dimensional analysis gives $r_0 \propto M_\star^{1/3}$ with **zero free exponent**.
The normalisation is fitted once from the full SPARC sample.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

M_stars = np.array([3e7,  4.5e9, 3.5e10, 2e10,  2e11])
r0_vals = np.array([0.10, 2.3,   6.7,    7.8,   17.7])
names   = ['DDO 154', 'NGC 2403', 'NGC 3198', 'NGC 6503', 'UGC 2885']
chi2s   = [3.55, 1.40, 1.38, 12.75, 8.08]

logM = np.linspace(6.5, 11.5, 100)
b = np.log10(6.7) - (1/3)*np.log10(3.5e10)

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(10**logM, 10**((1/3)*logM + b), 'g-', lw=2,
           label=r'ECT Level A: $r_0\propto M_\star^{1/3}$')
sc = ax.scatter(M_stars, r0_vals, c=chi2s, cmap='RdYlGn_r',
                s=100, zorder=5, vmin=1, vmax=13)
plt.colorbar(sc, ax=ax, label=r'$\chi^2/N$')
for i, n in enumerate(names):
    ax.annotate(n, (M_stars[i], r0_vals[i]),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel(r'$M_\star\;[M_\odot]$', fontsize=12)
ax.set_ylabel(r'$r_0$ [kpc]', fontsize=12)
ax.set_title('ECT condensate scale vs stellar mass\n(colour = fit quality)', fontsize=12)
ax.legend(fontsize=11); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()
print('Full 175-galaxy SPARC fit: see ect_sparc_results_v3.csv')
print('Milky Way: r0=5.7 kpc, chi2/N=2.84 (ECT) vs 3.51 (LCDM/NFW, 3 params)')